<a href="https://colab.research.google.com/github/camiloch28/everpeak-analysis/blob/main/everpeak-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisis Everpeak

## 1. Identificando variables relevantes para el analisis.

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# cargar archivo
df = pd.read_csv('/datasets/everpeak_retail.csv')

In [ ]:
# Cardinalidad, entender la estructura del dataset
df.nunique().sort_values()

In [ ]:
# Identificar valores faltantes
df.isna().mean().sort_values(ascending = False)

In [ ]:
# Identificar valores invalidos

## Revisar categorias invalidas
df["product_category"].value_counts()
df[df["product_category"].isin("?")]

## Revisar cantidades menores a cero
df["quantity"].le(0).sum()

## Revisar edades improbables
df[["customer_age","quantity"]].describe()
df["customer_age"].isin([-999]).sum()

#2. Manejando valores ausentes o invalidos

In [ ]:
# Que tipo de valor ausente es customer_age

## Agrupar por ciudad para saber si existe una relacion entre la ciudad y la edad
df.["customer_age"].isna().groupby(df["city"]).mean().sort_values(ascending = False)

## Agrupar por categoria de producto para ver si existe una relacon
df.["customer_age"].isna().groupby(df["product_category"]).mean()

## Crear Flag de fila vacia para validar si la variable es especial
df["edad_vacia"] = df["customer_age"].isna().astype(int)

## Comparar order_value
df.groupby("edad_vacia")["order_value"].describe()

#3. Imputar costumer_age

In [ ]:
## Calcular la mediana
median_age = df["costumer_age"].median()
print(median_age)

## Imputar con la mediana
df["customer_age"] = df["customer_age"].fillna(median_age, inplace = True)

## Por ultimo validamos que la imputacion funciono correctamente
df["customer_age"].isna().sum() #Se deberia ver un cero como resultado

#4. Como limpiar quantity menor o igual a cero

quantity = order_value/price

In [ ]:
## Revisar valores invalidos en columna de cantidad (quantity)
df["quantity"].le(0).sum()

## Marcar como NaN los valores invalidos de quantity
df.loc[df["quantity"] <= 0, "quantity"] = np.nan

## Validar si la cantidad calculada coincide con la columna quantity
df["calculated_quantity"] = (df["order_value"] / df["price"]).round()

## Imputar los valores faltantesco el calculo de order_value/price
df["quantity"] = df["quantity"].fillna(df["order_value"] / df["price"])

## Revisar la imputacion
df["quantity"].le(0).sum()

**#O CONSTRUIR UN PIPELINE COMPLETO PARA LIMPIEZA DE DATOS**

#Función 1: Estandarizar sentinels y valores inválidos

In [ ]:
# crear función para estandarizar sentinels
def reemplazar_sentinels_global(df, numeric_cols, text_cols):
    numeric_sentinels = [-999, 999]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].replace(numeric_sentinels, pd.NA)

    text_sentinels = ["?"]
    for col in text_cols:
        df[col] = df[col].replace(text_sentinels, "unknown")
    return df

# establecer columnas a procesar
columnas_numericas = ["customer_age"]      # columnas numéricas a procesar
columnas_texto = ["product_category"]       # columnas texto a procesar

# aplicar función
reemplazar_sentinels_global(df, columnas_numericas, columnas_texto):

# mostrar resultados
df.info()
print(df.head())

Función 2: Crear flags antes de imputar

In [ ]:
# crear función para crear columnas flags
def crear_flags(df, flags_cols):
    for col in flags_cols:
        nombre_flag = col + "_missing_flag"
        df[nombre_flag] = df[col].isna().astype(int)
    return df

# establecer columnas a procesar
columnas_flags= ["customer_age", "city", "state"]

# aplicar función y mostrar resultados
df = crear_flags_antes_de_imputar(df, columnas_flags)
df.info()
print(df.head())


Función 3: Imputar valores ausentes según diagnóstico

In [ ]:
median_fill_cols =["customer_age"]

for col in median_fill_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    med = df[col].median()
    df[col] = df[col].fillna(med)

#Rellenar valores ausentes con unknown
fill_unknown_cols = ["city", "state"]   # lista de columnas a rellenar con unknown
for col in fill_unknown_cols:
    df[col] = df[col].fillna("unknown")

Esto se soluciona eliminando los valores ausentes en esa columna.

#Eliminar valores ausentes en una columna
for col in date_drop_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df = df.dropna(subset=[col]).reset_index(drop=True)
    return df

#Codigo para formato fecha
date_drop_cols= ["order_date"]

for col in date_drop_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")
    df = df.dropna(subset=[col]).reset_index(drop=True)


# main_pipeline.py

# DataFrame a procesar: Everpeak
df = pd.read_csv("/datasets/everpeak_retail.csv")

# Listas de columnas - incluir columnas según las necesidades del dataset

# columnas a procesar del dataset EverPeak
## columnas función 1: reemplazar_sentinels
columnas_numericas = ["customer_age"]
columnas_texto = ["product_category"]

## columnas función 2: crear_flags
columnas_flags = ["customer_age", "city", "state"]

## columnas función 3: imputar_segun_diagnostico
cols_imputar_mediana = ["customer_age"]
cols_imputar_unknown = ["city", "state"]
cols_imputar_fecha = ["order_date"]

# aplicar función y guardar resultado en df_clean
df_clean = clean_data(df, columnas_numericas, columnas_texto, columnas_flags,
    cols_imputar_mediana, cols_imputar_unknown, cols_imputar_fecha)

# guardar df_clean en un CSV nuevo
df_clean.to_csv("/datasets/everpeak_clean.csv", index=False)

#2.1. Medidas estadisticas en columnas numerica

1. Resumen numérico
Objetivo: El equipo de Inteligencia Comercial necesita una visión rápida del desempeño de las categorías Fashion y Sports para entender su situación actual y detectar posibles anomalías en los datos.


In [ ]:
# Crear dataframes para cada categoría
df_fashion = df[df['product_category'] == 'Fashion']
df_sports = df[df['product_category'] == 'Sports']

# --- Resumen de columnas numéricas con describe()
columnas_numericas = ['order_value', 'customer_age', 'price', 'quantity']

print('Resumen estadístico de la categoría Fashion')
print(df_fashion[columnas_numericas].describe())

print()#Salto de Línea
print('Resumen estadístico de la categoría Sports')
print(df_sports[columnas_numericas].describe())

2. Promedio vs Mediana del gasto (Grocery)
Objetivo:

El equipo quiere entender si el gasto típico de los clientes que compran productos de supermercado está siendo afectado por outliers. Tu misión es comparar la media y la mediana de la columna order_value para evaluar si hay valores extremos influyendo en el análisis.


In [ ]:
# Filtrar la categoría Grocery
df_grocery = df[df['product_category'] == 'Grocery']

# Calcular media y mediana del gasto
promedio = df_grocery['order_value'].mean()
mediana = df_grocery['order_value'].median()

# Mostrar resultados
print("Promedio del gasto en Grocery:", promedio)
print("Mediana del gasto en Grocery:", mediana)

# Interpretación según comparación de media y mediana
print("El promedio está afectado por outliers o valores atípicos en Grocery.")


3.  Promedio vs Mediana
Objetivo: El equipo sospecha que existen outliers en la cantidad de productos comprados.


In [ ]:
# Promedio y mediana de quantity
print("Promedio de quantity: ", df['quantity'].mean())
print("Mediana de quantity: ", df['quantity'].median())
print("El promedio está afectado por los outliers o valores atípicos.")


4. Resumen numérico por ciudad
El equipo necesita una visión rápida del comportamiento de los clientes en las ciudades New York y Los Angeles, para comprender su situación actual y detectar posibles anomalías en los datos.


In [ ]:
# Crear dataframes para cada categoría
df_ny = df[df["city"] == 'New York']
df_la = df[df["city"] == 'Los Angeles']

# --- Resumen de columnas numéricas con describe()
columnas_numericas = ['order_value', 'customer_age', 'price', 'quantity']

print('Resumen estadístico de la ciudad New York')
print(df_ny[columnas_numericas].describe())

print()#Salto de Línea
print('Resumen estadístico de la ciudad Los Angeles')
print(df_la[columnas_numericas].describe())

#2.2. Medidas estadisticas en columnas categoricas

In [ ]:
#Analicemos las columnas categóricas del dataset

# Frecuencia absoluta y relativa de product_category
print("Frecuencia absoluta:")
print(df['product_category'].value_counts())

print("\nFrecuencia relativa:")
print(df['product_category'].value_counts(normalize=True))

# Columnas categóricas
columnas_categoricas = ['product_category', 'payment_method', 'city', 'state']

# Resumen con describe()
print(df[columnas_categoricas].describe())

#Luego, para profundizar en todas las categorías y su frecuencia, usamos value_counts(). Esto nos permite calcular la frecuencia absoluta y frecuencia relativa para cada categoría:

for col in columnas_categoricas:
    print(col)
    print("Frecuencia absoluta:")
    print(df[col].value_counts())
    print("\nFrecuencia relativa:")
    print(df[col].value_counts(normalize=True))
    print("")

#2.3. Visualizando distribuciones con Histogramas

In [ ]:
#Con la libreria matplotlib.pyplot

plt.hist(df['score'], bins=10, color='skyblue', edgecolor='black')

plt.xlabel('Notas de los estudiantes')
plt.ylabel('Cantidad de estudiantes')
plt.title('Distribución de notas - Matplotlib')
plt.show()

#Con la libreria Seaborn

sns.histplot(df['score'], bins=10, color='skyblue', kde=True)

plt.xlabel('Notas de los estudiantes')
plt.ylabel('Cantidad de estudiantes')
plt.title('Distribución de notas - Seaborn')
plt.show()


#2.4. Explorando distribuciones con Boxplots e Histogramas

In [ ]:
sns.boxplot(x=df['order_value'], color='skyblue')
plt.title('Boxplot de order_value')
plt.xlabel('Order value')
plt.show()

#3.1 Identificando valores atípicos con reglas estadísticas

In [ ]:
#calcular Q1
Q1 = df['price'].quantile(0.25)
print('Primer cuartil: ', Q1)

#calcular Q3
Q3 = df['price'].quantile(0.75)
print('Tercer cuartil: ', Q3)
Ya que tenemos los cuantiles, podemos obtener el rango intercuartílico.

#calcular IQR
IQR = Q3 - Q1
print('IQR: ', IQR)

#calcular límite inferior
lower = Q1 - 1.5 * IQR
print('Límite inferior: ', lower)

#calcular límite superior
upper = Q3 + 1.5 * IQR
print('Límite superior: ', upper)

#El puntaje Z, o Z-Score.

In [ ]:
#calculamos el promedio
mean = df['price'].mean()
Después calculamos la desviación estándar con std()


#calculamos la desviación estándar
std = df['price'].std()
Y finalmente aplicamos la fórmula:

#calculamos el valor z en una nueva columna aplicando la fórmula
df['z'] = (df['price'] - mean) / std

#vemos registros donde el valor Z del precio sea mayor a 3
df[df['z'].abs() > 3]

#verificamos
df['z'].describe()

#3.2. Cómo abordar valores atípicos según el contexto

In [ ]:
#calculo percentil 1 y 99
lower = df['order_value'].quantile(0.01)
lower

upper = df['order_value'].quantile(0.99)
upper

#reemplazo los valores
df['order_value_winsor'] = np.clip(df['order_value'], lower, upper)
df[["order_value", "order_value_winsor"]].head()



#3.3. Segmentación de clientes con sentencias if

In [ ]:
# Calcular promedio y mediana
cantidad_promedio = df['quantity'].mean()
cantidad_mediana = df['quantity'].median()
print("Promedio:", cantidad_promedio)
print("Mediana:",cantidad_mediana)
print()

# Segmentación con media o promedio
if cantidad_promedio > 22:
	print("En promedio: volumen alto")
elif cantidad_promedio >= 10:
	print("En promedio: volumen medio")

else:
	print("En promedio: volumen bajo")

# Segmentación con mediana
if cantidad_mediana > 22:
	print("Según la mediana: volumen alto")
elif cantidad_mediana >= 10:
	print("Según la mediana: volumen medio")

else:
	print("Según la mediana: volumen bajo")

#3.4. Segmentación para el análisis de clientes

In [ ]:
# Función para clasificar clientes
def classify_segment(row):
     age = row['customer_age']
    spend = row['order_value']

    # Manejo de valores nulos/faltantes
    # pd.isna() verifica de forma robusta si el valor es NaN
    if pd.isna(age) or pd.isna(spend):
        return "Error en Datos"

    # Segmentación de Alto Valor (Gasto >= 10000)
    if spend >= 10000:
        if age >= 55:
            return "Senior VIP"
        else: # age < 55
            return "Junior VIP"

    # Segmentación de Valor Medio (Gasto entre 5000 y 9999)
    elif spend >= 5000:
        if age >= 55:
            return "Sr. Medium Value"
        else: # age < 55
            return "Jr. Medium Value"

    # Segmentación de Valor Bajo (Gasto < 5000)
    else: # spend < 5000
        return "Low Value"

# aplicar función y ver cambios
df["customer_segment"] = df.apply(classify_segment, axis=1)
df[["customer_age", "order_value", "customer_segment"]].head()

#3.5. Redacción de un resumen estadístico

✔ “¿Está la distribución sesgada?”
→ Si sí, ¿es un sesgo natural del retail? (ej. precios y order_value)

→ ¿O revela errores sistemáticos?

✔ “¿La media y la mediana dicen historias distintas?”
Si la media es mucho mayor o menor que la mediana → sesgo (skew) + outliers.

Si la media es similar a la mediana → distribución aproximadamente simétrica.

✔ “¿Qué outliers existen y qué tipo son?”
Errores imposibles → drop
Extremos reales → keep
Mezcla real + error (comportamiento típico) → winsorize
✔ “¿Qué tratamiento aplicaste y por qué?”
Explica la decisión en lenguaje técnico + empresarial.

✔ “¿Cómo cambian las estadísticas después del tratamiento?”
Es la prueba final de que tu decisión fue razonable.

✔ “¿Existen valores nulos o faltantes?”
Necesario para saber si debo tratarlos de alguna manera.

✔ “¿Hay valores duplicados?”
Un chequeo indispensable para evitar errores en los cálculos.

A continuación, crearemos un análisis estadístico.

Tomaremos el dataset limpio de EverPeak y analizaremos las columnas numéricas relevantes.

Análisis estadístico
# importar librerías y dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv("/datasets/everpeak_clean.csv")
Comenzamos con una revisión general de todas las variables.

Por ahora nos enfocaremos en las columnas numéricas.

Identificamos las columnas relevantes como: price, quantity, order_value y customer_age.
E incluimos una breve descripción de ellas y por qué las analizamos.
1. Visión general
df.info()
df.head()
Las columnas price, quantity, order_value y customer_age son de tipo INT.

Las analizamos porque permiten identificar patrones, distribuciones y valores atípicos útiles para el análisis.

Posteriormente, calculamos sus estadísticas descriptivas básicas:

Observamos la cantidad de valores, la media, el mínimo y máximo, los cuartiles y la mediana.
2. Estadísticas descriptivas
num_cols = ["price", "quantity", "order_value", "customer_age"]
df[num_cols].describe()
Después pasamos a la visualización diagnóstica

Aquí, evaluamos la distribución de cada variable mediante un histograma y un boxplot, para ver su forma y detectar la presencia de sesgo.

En price, quantity y order_value se detectan valores atípicos hacia la derecha, mientras que en customer_age no se observan.

3. Visualización diagnóstica
# Graficar histogramas
for col in num_cols:
    plt.figure(figsize=(9, 3))
    sns.histplot(df[col], bins=60)
    plt.title(f'Distribución de la variable {col}')
    plt.show()
# Graficar boxplots
for col in num_cols:
    plt.figure(figsize=(9,3))
    sns.boxplot(x=df[col])
    plt.title(f'Distribución de la variable {col}')
    plt.show()
A continuación, pasamos a la identificación formal de outliers.

Además de identificarlos con el boxplot, debemos usar el IQR o z-score para realizar un doble chequeo y analizar los valores atípicos en caso de que existan.

4. Identificación formal de outliers
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    print(col, "IQR: ", IQR)
    
    upper = Q3 + 1.5*IQR
    lower = Q1 - 1.5*IQR
    display(df[(df['price'] > upper) | (df['price'] < lower)])
Si hay valores atípicos presentes, seguimos con la clasificación de outliers.

Para cada tipo de valor atípico debemos indicar si:

Se trata de un error y debe ser eliminado.
Si es un valor posible a mantener.
O, si debemos caparlo.
Recuerda indicar y justificar en cada caso qué haremos y por qué:

En este ejemplo, tenemos únicamente valores atípicos altos que son posibles pero poco comunes; para que el análisis represente al cliente típico, capamos los valores mayores al percentil del 99.

5. Clasificación y tratamiento de outliers: Drop, Keep o Cap
Drop: cuando el valor es imposible
Keep: es un valor posible a mantener
Winsorization: cuando es extremo pero posible
5.1 Winsorization (valores extremos pero posibles)
Para conservar la información del cliente típico, capamos los valores que estén por encima del percentil del 99.

for col in num_cols:
    p99 = df[col].quantile(0.99)
    df[f'{col}_capped'] = np.clip(df[col], None, p99)

df[["price","price_capped", "quantity","quantity_capped",
    "order_value","order_value_capped", "customer_age","customer_age_capped"]].head()
Antes de terminar, es importante resaltar las estadísticas post-tratamiento.

Esto nos permite comparar el antes y el después, destacando ventajas, inferencias y posibles puntos de conflicto.

6. Estadísticas post-tratamiento
df[["price","price_capped", "quantity","quantity_capped",
    "order_value","order_value_capped", "customer_age","customer_age_capped"]].describe()
Finalmente, cerramos con una conclusión ejecutiva para cada característica, es decir una frase clara y comprensible para el área de negocio, tal como se muestra en el ejemplo en pantalla.

7. Conclusión ejecutiva
7.1 Columna price:
El 50% de los precios se encuentra entre 218 y 847.
Los valores >847 USD corresponden al 25% de los precios más altos.
Se realizó winsorización al percentil 99 para estabilizar métricas sin perder estos segmentos.
Con esta estructura, tu análisis será claro, completo y fácil de interpretar.